# Content-Based Movie Recommendation System

## Business Scenario

You have been hired as a Data Scientist at **Netflix**. Netflix has thousands of movies and millions of users, and users spend too much time searching for something to watch — many leave the platform before selecting anything.

Netflix wants to recommend movies each user is likely to enjoy. In this notebook we tackle one piece of that puzzle: a **content-based recommender**. Given a movie a user already likes, the system suggests other movies with *similar content* (in this case, similar plot descriptions).

### Why Content-Based Filtering?

Content-based recommenders work by comparing the **attributes of the items themselves** (here, the text of each movie's overview) rather than relying on other users' behavior. This makes them especially useful because:

- They don't suffer from the "new item" cold-start problem — a brand-new movie can be recommended as soon as it has a description, even with zero ratings.
- Recommendations are easy to explain ("recommended because it's similar to X").
- They don't require any data about other users, which is useful when user-rating history is sparse.

The trade-off is that content-based systems only look at item similarity, not personal taste, so they are usually combined with collaborative filtering (see the companion notebook) in a real Netflix-style system.

## The Process

1. Load the movie metadata
2. Handle missing values
3. Convert plot overviews into TF-IDF vectors
4. Compute cosine similarity between movies
5. Build a title-to-index lookup
6. Write a recommendation function
7. Test the recommender on example movies
8. Discuss strengths, limitations, and next steps


## Step 1: Load the data

In [1]:
import pandas as pd

# low_memory=False prevents Pandas from incorrectly guessing data types when reading a large file.
movies = pd.read_csv("movies_metadata.csv", low_memory=False)

# For a content-based recommender we only need the title and the plot overview —
# these are the two columns that describe *what the movie is about*.
movies = movies[['title', 'overview']]

movies.head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


## Step 2: Handle missing values

In [2]:
movies['overview'] = movies['overview'].fillna('')

# The fillna('') method replaces every missing overview with an empty string.
# This is necessary because the TF-IDF vectorizer cannot process missing (NaN) values —
# a movie with no overview simply contributes no words to the vocabulary.

## Step 3: Convert overviews into TF-IDF vectors

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['overview'])

# TfidfVectorizer converts text into numerical features.
# stop_words='english' removes common words such as "the", "is", and "and" because they provide little useful information.
# fit_transform(): converts each overview into a numerical vector based on TF-IDF scores.
# tfidf_matrix: each row represents a movie, each column represents a unique word,
# each value represents how important that word is to that movie's overview
# (relative to how common it is across all overviews).
# This turns unstructured plot text into a numerical representation that can be compared mathematically.

print(f"TF-IDF matrix shape: {tfidf_matrix.shape[0]} movies x {tfidf_matrix.shape[1]} vocabulary terms")

TF-IDF matrix shape: 45466 movies x 75827 vocabulary terms


## Step 4: Compute similarity between movies

Cosine similarity measures how similar two movies are based on their TF-IDF vectors — the smaller the angle between two vectors, the more similar the movies' overviews are.

**A note on scale:** with ~45,000 movies, computing and storing the *full* pairwise similarity matrix up front (movie x movie) would require a 45,466 x 45,466 matrix — over 15 GB of memory. That doesn't scale, and it's wasteful, since we only ever need the similarity scores for **one movie at a time** (whichever movie the user is currently looking at).

Instead, we compute the similarity row for a single movie against every other movie **on demand**, right when a recommendation is requested. This produces mathematically identical results to computing the full matrix, but only uses a tiny fraction of the memory — a design choice that matters even more at Netflix's real scale of millions of titles and users.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similarity_scores(movie_idx, tfidf_matrix=tfidf_matrix):
    """Compute cosine similarity between one movie and every other movie.

    Only computes a single row of the similarity matrix (1 x n_movies) instead of the
    full (n_movies x n_movies) matrix, which keeps memory usage low.
    """
    return cosine_similarity(tfidf_matrix[movie_idx], tfidf_matrix).flatten()

# Quick sanity check: a movie should be perfectly similar to itself (score of 1.0)
sample_scores = get_similarity_scores(0)
print(f"Similarity of 'Toy Story' with itself: {sample_scores[0]:.2f}")

Similarity of 'Toy Story' with itself: 1.00


## Step 5: Create an index of movie titles

In [5]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# This creates a mapping between movie titles and their row indices in the DataFrame.
# Allows the recommender to quickly locate a movie using its title.
# drop_duplicates() keeps the first occurrence when the same title appears more than once
# (e.g. remakes), so title lookups always resolve to a single row.

## Step 6: Recommendation function

In [6]:
def content_recommender(title, n=10):
    """Recommend movies with the most similar plot overview to the given title."""

    if title not in indices:
        print(f"'{title}' was not found in the dataset.")
        return None

    idx = indices[title]  # Find the movie's row index

    sim_scores = get_similarity_scores(idx)  # Similarity of this movie against all others
    sim_scores = list(enumerate(sim_scores))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)  # Sort by similarity

    sim_scores = sim_scores[1:n + 1]  # Skip index 0 (the movie itself) and keep the top n

    movie_indices = [i[0] for i in sim_scores]  # Retrieve movie indices
    scores = [i[1] for i in sim_scores]

    recommendations = movies.iloc[movie_indices][['title']].copy()
    recommendations['similarity_score'] = scores

    return recommendations

# Generates a list of movies with descriptions most similar to the movie selected by the user.

## Step 7: Test it

In [7]:
content_recommender("Toy Story")

,title,similarity_score
15348,Toy Story 3,0.531909
2997,Toy Story 2,0.471984
10301,The 40 Year Old Virgin,0.275047
24523,Small Fry,0.273062
23843,Andy Hardy's Blonde Trouble,0.235495
29202,Hot Splash,0.223841
43427,Andy Kaufman Plays Carnegie Hall,0.217688
38476,Superstar: The Life and Times of Andy Warhol,0.215959
42721,Andy Peters: Exclamation Mark Question Point,0.201972
8327,The Champ,0.198825


In [8]:
content_recommender("Jumanji")

,title,similarity_score
21633,Table No. 21,0.228063
45253,Quiz,0.191327
41573,Snowed Under,0.189125
35509,The Mend,0.188841
44376,Liar Game: Reborn,0.187119
17223,The Dark Angel,0.179439
8801,Quintet,0.177905
6166,Brainscan,0.177169
30981,Turkey Shoot,0.171180
9503,Word Wars,0.169356


In [9]:
# A quick test of the "not found" path, so we know the function fails gracefully
# for a movie title that isn't in our catalog.
content_recommender("A Movie That Does Not Exist")

'A Movie That Does Not Exist' was not found in the dataset.


## Step 8: Discussion — Strengths, Limitations, and Next Steps

**How well did it work?**
For "Toy Story", the top results include its direct sequels ("Toy Story 2" and "Toy Story 3") — a strong sign the model is picking up genuinely relevant similarity from plot text alone. For "Jumanji", the results are noticeably weaker and less intuitive (e.g. "Table No. 21", "Quiz"). This is expected: TF-IDF only matches on shared vocabulary, so it can be fooled by unrelated movies that happen to share incidental words, and it has no understanding of genre, tone, or theme beyond the words used in the overview.

**Strengths of this approach:**
- No cold-start problem for new movies — any title with a written overview can be recommended immediately.
- Fully explainable: recommendations can be justified by shared plot vocabulary.
- Doesn't require any user rating history at all.

**Limitations:**
- Recommendations are entirely user-agnostic — two very different Netflix users who both watched "Toy Story" get the exact same suggestions.
- TF-IDF only captures shared words, not deeper semantic meaning (e.g. it won't recognize that two movies are both "heartwarming family comedies" if they use different vocabulary to say so).
- Overviews are short, so the similarity signal is noisy; richer features (genre, cast, director, keywords) would likely improve quality.

**Next steps for a production Netflix-style system:**
- Enrich the feature set beyond `overview` alone (genres, cast, director, keywords) to build a stronger content profile per movie.
- Combine this content-based recommender with the **collaborative filtering** model (see the companion notebook) into a hybrid system — content-based for cold-start items and new users, collaborative filtering once enough rating data exists.
- Evaluate quality with real user feedback (click-through / watch-through rate) in addition to eyeballing similarity scores, since there is no ground-truth "correct" recommendation list to compute an offline accuracy metric against.